In [2]:
!pip install gsw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 194.4 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 212.4 kB/s  0:00:25m0:00:0100:02
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gsw]1/2 [gsw]


In [ ]:
import os
import gsw
import numpy as np
import pandas as pd


def generate_float_level_climatologies_300m():
    """Aggregates raw oceanographic profile timeseries into float-level baselines down to 300m.

    This function reads raw profile measurements from autonomous BGC-Argo floats,
    computes potential density anomalies using TEOS-10 formulation, calculates individual 
    profile boundaries (Mixed Layer Depth and Deep Chlorophyll Maximum), tracks mean 
    spatial positions, and compiles a clean depth-stratified climatological dataset.

    Required Input Columns in 'raw_monthly_regional_profiles.csv':
    -------------------------------------------------------------
    - Region (str): Basin (e.g., 'Arabian Sea', 'Bay of Bengal').
    - WMO_ID (int/str): World Meteorological Organization unique identifier for the float.
    - Profile_Date (str/datetime): Timestamp of when the individual profile was taken.
    - Depth_m (float): Water depth down to which measurements were taken (meters).
    - Temperature_C (float): In-situ water temperature (degrees Celsius).
    - Salinity_psu (float): Practical salinity (psu).
    - Chl_a (float): Chlorophyll-a concentration (ug/L).
    - Nitrate (float): Nitrate concentration (umol/kg).

    Required Input Columns in 'bgc_argo_processed_profiles.csv':
    ------------------------------------------------------------
    - WMO_ID or WMO (int/str): Unique float identifier used to cross-match spatial positions.
    - Latitude (float): Geolocation coordinate in decimal degrees.
    - Longitude (float): Geolocation coordinate in decimal degrees.

    Algorithmic Methodology:
    ------------------------
    1. Potential Density Anomaly (\sigma_\theta): Calculated via `gsw.sigma0` using practical 
       salinity and in-situ temperature at atmospheric pressure (0 dbar anomaly).
    2. Mixed Layer Depth (MLD): Calculated using a threshold criterion. The algorithm sets a 
       near-surface reference depth at 10m to avoid diurnal skin-layer artifacts. It searches 
       downward for the first depth where potential density increases by >= 0.125 kg/m³ relative 
       to the 10m reference. If no threshold is met, the maximum profile depth is assigned.
    3. Deep Chlorophyll Maximum (DCM): Isolated by finding the absolute mathematical maximum 
       of the Chl-a channel within the photic zone ecosystem envelope (10m to 200m depth). A baseline 
       noise floor threshold (> 0.01 ug/L) is enforced to ignore null biomes.
    4. Temporal & Spatial Averaging: Individual profile features are averaged per float to 
       extrapolate a representative baseline. Water column variables are grouped by Float and 
       Depth step, computing a clean climatological mean signature.

    Output:
    -------
    Saves a master aggregated CSV matrix to disk containing columns:
    ['Region', 'WMO_ID', 'Depth_m', 'Temperature_C', 'Salinity_psu', 'Chl_a', 
     'Nitrate', 'MLD', 'DCM', 'Latitude', 'Longitude']
    Where columns represent the depth-stratified float summary mapping physical structures 
    and chemical properties down to a standardized 300-meter threshold.
    """
    print("--- Aggregating Float Profiles down to 300m ---")

    input_csv = (
        "/Users/ishita/Desktop/bioargov2/raw_monthly_regional_profiles.csv"
    )
    source_coords_csv = (
        "/Users/ishita/Desktop/bioargov2/bgc_argo_processed_profiles.csv"
    )

    if not os.path.exists(input_csv) or not os.path.exists(source_coords_csv):
        print("Error: Missing required input CSV files.")
        return

    df = pd.read_csv(input_csv)
    df["Sigma_Theta"] = gsw.sigma0(df["Salinity_psu"], df["Temperature_C"])

    print("Calculating boundaries for individual profiles...")
    profile_boundaries = []
    profile_groups = df.groupby(["Region", "WMO_ID", "Profile_Date"])

    for (region, wmo_id, profile_date), group in profile_groups:
        if group.empty:
            continue

        prof = group.sort_values(by="Depth_m")
        d = prof["Depth_m"].values
        rho = prof["Sigma_Theta"].values
        chl = prof["Chl_a"].values

        mld = np.nan
        dcm = np.nan

        if len(d) > 1:
            ref_idx = np.abs(d - 10.0).argmin()
            threshold_rho = rho[ref_idx] + 0.125
            valid_mld = [
                i for i, r in enumerate(rho) if i != ref_idx and r >= threshold_rho
            ]
            mld = d[valid_mld[0]] if valid_mld else np.max(d)

        # Look down to 200m for DCM to handle deep photic zones safely
        dcm_zone = (d >= 10) & (d <= 200)
        if np.any(dcm_zone):
            d_sub, chl_sub = d[dcm_zone], chl[dcm_zone]
            if np.max(chl_sub) > 0.01:
                dcm = d_sub[chl_sub.argmax()]

        profile_boundaries.append(
            {
                "Region": region,
                "WMO_ID": wmo_id,
                "Profile_Date": profile_date,
                "MLD": mld,
                "DCM": dcm,
            }
        )

    prof_bound_df = pd.DataFrame(profile_boundaries)
    float_boundaries = (
        prof_bound_df.groupby(["Region", "WMO_ID"])[["MLD", "DCM"]]
        .mean()
        .reset_index()
    )

    print("Averaging water column variables per float...")
    float_profiles = (
        df.groupby(["Region", "WMO_ID", "Depth_m"])[
            ["Temperature_C", "Salinity_psu", "Chl_a", "Nitrate"]
        ]
        .mean()
        .reset_index()
    )

    src_df = pd.read_csv(source_coords_csv)
    src_df.columns = src_df.columns.str.strip()

    lat_col = [c for c in src_df.columns if c.lower() == "latitude"][0]
    lon_col = [c for c in src_df.columns if c.lower() == "longitude"][0]
    wmo_col = [c for c in src_df.columns if c.lower() in ["wmo_id", "wmo"]][0]

    coords_mapped = (
        src_df.groupby(wmo_col)[[lat_col, lon_col]].mean().reset_index()
    )
    coords_mapped.columns = ["WMO_ID", "Latitude", "Longitude"]

    final_float_ds = pd.merge(
        float_profiles, float_boundaries, on=["Region", "WMO_ID"], how="left"
    )
    final_float_ds = pd.merge(
        final_float_ds, coords_mapped, on="WMO_ID", how="left"
    )

    output_csv = (
        "/Users/ishita/Desktop/bioargov2/float_level_profile_averages_300m.csv"
    )
    final_float_ds.to_csv(output_csv, index=False)
    print(f"Data compiled to: {output_csv}")


if __name__ == "__main__":
    generate_float_level_climatologies_300m()

--- Aggregating Float Profiles down to 300m ---
Calculating boundaries for individual profiles...
Averaging water column variables per float...
Success! Data compiled to: /Users/ishita/Desktop/bioargov2/float_level_profile_averages_300m.csv


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

input_csv = "/Users/ishita/Desktop/bioargov2/float_level_profile_averages_300m.csv"
output_dir = "/Users/ishita/Desktop/bioargov2/5 variables plots"

# 1. Clear out any older plots in the folder first
print("--- Cleaning Folder & Re-plotting MLD/DCM Profiles ---")
if os.path.exists(output_dir):
    for filename in os.listdir(output_dir):
        if filename.endswith(".png"):
            os.remove(os.path.join(output_dir, filename))
else:
    os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_csv)
unique_floats = df["WMO_ID"].unique()

print(f"Processing {len(unique_floats)} individual float profiles...")

for wmo_id in unique_floats:
    float_df = df[df["WMO_ID"] == wmo_id].sort_values(by="Depth_m")
    float_df = float_df[float_df["Depth_m"] <= 300]  # Focus strictly down to 300m
    
    if float_df.empty:
        continue

    region = float_df["Region"].iloc[0]
    depths = float_df["Depth_m"].values
    chl = float_df["Chl_a"].values
    temp = float_df["Temperature_C"].values
    sal = float_df["Salinity_psu"].values
    nitrate = float_df["Nitrate"].values

    mean_lat = float_df["Latitude"].iloc[0] if "Latitude" in float_df else np.nan
    mean_lon = float_df["Longitude"].iloc[0] if "Longitude" in float_df else np.nan
    mld_val = float_df["MLD"].iloc[0]
    dcm_val = float_df["DCM"].iloc[0]

    # Create an independent figure frame for this float
    fig, ax_chl = plt.subplots(figsize=(8, 10))

    # --- MAIN BOTTOM AXIS: Chlorophyll-a ---
    line_chl = ax_chl.plot(chl, depths, color="green", linewidth=2.5, label="Chl-a")
    ax_chl.set_xlabel(r"Chlorophyll-a ($\mu g / L$)", color="green", fontsize=11, fontweight="bold")
    ax_chl.tick_params(axis="x", labelcolor="green")
    ax_chl.grid(True, linestyle=":", alpha=0.5)
    ax_chl.set_ylabel("Depth (meters)", fontsize=12, fontweight="bold")

    # Dynamically pad the Chl axis based on its own ranges
    if len(chl) > 0 and not np.all(np.isnan(chl)):
        ax_chl.set_xlim(np.nanmin(chl) - 0.05, np.nanmax(chl) + 0.1)

    # --- TOP TWINY AXIS 1: Temperature ---
    ax_temp = ax_chl.twiny()
    line_temp = ax_temp.plot(temp, depths, color="crimson", linewidth=1.8, label="Temp")
    ax_temp.set_xlabel("Temperature (°C)", color="crimson", fontsize=11)
    ax_temp.tick_params(axis="x", labelcolor="crimson")
    if len(temp) > 0 and not np.all(np.isnan(temp)):
        ax_temp.set_xlim(np.nanmin(temp) - 1, np.nanmax(temp) + 1)

    # --- TOP TWINY AXIS 2: Salinity (Displaced Upward) ---
    ax_sal = ax_chl.twiny()
    ax_sal.spines["top"].set_position(("outward", 40))
    line_sal = ax_sal.plot(sal, depths, color="blue", linewidth=1.8, linestyle="-", label="Sal")
    ax_sal.set_xlabel("Salinity (psu)", color="blue", fontsize=11)
    ax_sal.tick_params(axis="x", labelcolor="blue")
    if len(sal) > 0 and not np.all(np.isnan(sal)):
        ax_sal.set_xlim(np.nanmin(sal) - 0.2, np.nanmax(sal) + 0.2)

    # --- TOP TWINY AXIS 3: Nitrate (Displaced Further Upward) ---
    ax_nit = ax_chl.twiny()
    ax_nit.spines["top"].set_position(("outward", 80))
    line_nit = ax_nit.plot(nitrate, depths, color="darkorange", linewidth=1.8, label="Nitrate")
    ax_nit.set_xlabel(r"Nitrate ($\mu mol / kg$)", color="darkorange", fontsize=11)
    ax_nit.tick_params(axis="x", labelcolor="darkorange")
    if len(nitrate) > 0 and not np.all(np.isnan(nitrate)):
        ax_nit.set_xlim(np.nanmin(nitrate) - 1, np.nanmax(nitrate) + 2)

    # --- REFERENCE BOUNDARIES: Dotted Lines Only ---
    if not pd.isna(mld_val):
        ax_chl.axhline(y=mld_val, color="black", linestyle=":", linewidth=2)
    if not pd.isna(dcm_val):
        ax_chl.axhline(y=dcm_val, color="darkgreen", linestyle=":", linewidth=2)

    # Invert Y-axis so depth increases downward
    ax_chl.set_ylim(0, 300)
    ax_chl.invert_yaxis()

    # Dynamic Location Coordinates Configuration
    if not pd.isna(mean_lat) and not pd.isna(mean_lon):
        pos_text = (
            f"Avg Position: {abs(mean_lat):.2f}°{'N' if mean_lat >= 0 else 'S'}, "
            f"{abs(mean_lon):.2f}°{'E' if mean_lon >= 0 else 'W'}"
        )
    else:
        pos_text = "Position Unknown"

    # Build Header Title
    title_string = f"Float: {wmo_id} | Basin: {region}\n{pos_text}"
    ax_chl.set_title(title_string, pad=115, fontsize=12, fontweight="bold")

    # Consolidate Legends
    handles = (
        line_chl + line_temp + line_sal + line_nit
        + [
            plt.Line2D([0], [0], color="black", linestyle=":"),
            plt.Line2D([0], [0], color="darkgreen", linestyle=":"),
        ]
    )
    labels = [
        "Chlorophyll-a", "Temperature", "Salinity", "Nitrate",
        f"Avg MLD ({mld_val:.1f}m)", f"Avg DCM ({dcm_val:.1f}m)"
    ]
    ax_chl.legend(handles, labels, loc="lower left", fontsize=9, frameon=True, facecolor="white", framealpha=0.9)

    # Save to file path
    file_name = f"mld_dcm_profile_{wmo_id}.png"
    plt.savefig(os.path.join(output_dir, file_name), dpi=300, bbox_inches="tight")
    plt.close()

print(f"Profiles exported to: '{output_dir}'")

--- Cleaning Folder & Re-plotting MLD/DCM Profiles ---
Processing 28 individual float profiles...
Success! Standalone customized profiles exported to: '/Users/ishita/Desktop/bioargov2/5 variables plots'
